# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Content Refresh Prioritization, which maps onto a Supervised Scoring and Ranking task. While identifying whether a page is decaying looks like binary classification (Declining: Yes/No), our operational constraint means we cannot act on every single page at once. Therefore, we frame this as a scoring task where the model outputs a continuous probability score of decay for each page, allowing us to generate a globally ranked queue.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The actual outcome we wish to optimize is 'relevance or hidden decay', which cannot be directly measured by software. Instead, we use a proxy label: trend_direction == 'down'. This proxy tracks a sustained decline in impressions and search click-through performance over a 90-day window, serving as our measurable target column.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

The core business success metric is Precision@50. Because our human content team only has the capacity to review and rewrite 50 pages per week, we do not care about overall model accuracy or recall across the entire dataset. We care exclusively about the top 50 slots: of the top 50 highest-scored pages the model tells us to refresh first, how many are actually declining? Our baseline hand-written rule yields a Precision@50 of 0.240, while the initial Random Forest pipeline raises this to 0.740.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import os
import sys
import subprocess
import pandas as pd

# 1. Seamlessly setup workspace environment for Colab/Local execution
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    print("Cloning repository for data baseline sync...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# 2. Locate data asset dynamically
target_file = None
for root, dirs, files in os.walk("."):
    if "content_refresh_anonymized.csv" in files:
        target_file = os.path.join(root, "content_refresh_anonymized.csv")
        break

if not target_file:
    raise FileNotFoundError("Target dataset 'content_refresh_anonymized.csv' could not be resolved.")

# 3. Load dataframe and establish unit of analysis
df = pd.read_csv(target_file)

print("UNIT OF ANALYSIS SPECIFICATION:")
print(f"One Row = One Unique Web Page URL per client domain.\n" + "="*60)

# 4. Display a clean profile of features mapped against our target proxy label
feature_columns = ["search_volume", "avg_position", "impressions_90d", "ctr", "word_count", "trend_direction"]
preview_df = df[feature_columns].head(5).copy()

# Sketch the clear mathematical binary target vector from our proxy string
preview_df["target_is_declining"] = (preview_df["trend_direction"] == "down").astype(int)

# Render sample dataframe view
print(preview_df.to_string(index=False))

UNIT OF ANALYSIS SPECIFICATION:
One Row = One Unique Web Page URL per client domain.
 search_volume  avg_position  impressions_90d  ctr  word_count trend_direction  target_is_declining
          10.0          10.6             3803 0.76      3221.0            down                    1
          90.0          20.3            15320 0.05      2481.0            down                    1
           0.0          36.5            12581 0.09      3515.0            down                    1
          10.0           6.2            11751 0.49         NaN          stable                    0
           0.0          44.0            19140 0.13      2803.0            down                    1


As demonstrated in the dataframe output above, the unit of analysis is a single unique web page. Each row represents a page's historical performance metrics (features) linked to its empirical outcome status (trend_direction). The column target_is_declining showcases our engineered binary target vector where 1 represents an active decaying page requiring operational action.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed hardcoded rule operates like a rigid thermostat threshold (e.g., 'flag a page if CTR drops below 2% and average position falls by more than 3 places'). This approach fails fundamentally in organic search environments. Search patterns are multi-dimensional, non-linear, and vary drastically depending on user intent types, absolute search volumes, and domain scale.

A fixed rule cannot weigh the interaction between thousands of keyword variations and shifting position tiers simultaneously. Machine Learning beats a simple rule because it scales past hard thresholds, discovering the complex relational weights across all 44 columns to accurately surface hidden decay patterns before a catastrophic drop in traffic occurs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.